# Baseline Benchmark

Runs the initial LLM call on all benchmark queries, evaluates the results, and saves everything to disk (including `prompt.txt` per query so a refined run can load it later).

In [1]:
import importlib
import llm as llm_module
import filter as filter_module
importlib.reload(llm_module)
importlib.reload(filter_module)
from llm import call_llm, build_prompt
from filter import filter_services

import evaluate as evaluate_module
import analysis as analysis_module
from evaluate import evaluate, print_metrics, print_evaluation_summary
from benchmarks import load_benchmarks
from analysis import analyze
import output as output_module

## Configuration

In [2]:
from pathlib import Path
BENCHMARK_DIR = Path("./benchmark")
BENCHMARK_TYPES = ["restbench"]  # add more benchmark types here
BENCHMARK_LIMIT = None  # set to None for all sectors
QUERY_LIMIT = 50        # set to None for all queries
MAX_WORKERS = 10
MODEL = "deepseek-ai/DeepSeek-V4-Pro"

FILTER_SERVICES = True  # BM25 endpoint pre-filtering
TOP_K = 5               # endpoints to keep per service when FILTER_SERVICES=True

benchmark_sets, total_available, total_queries_available = load_benchmarks(BENCHMARK_DIR, BENCHMARK_TYPES, BENCHMARK_LIMIT, None)
total_queries = min(QUERY_LIMIT, total_queries_available) if QUERY_LIMIT is not None else total_queries_available
print(f"Loaded {len(benchmark_sets)} benchmark sets (total available: {total_available})")
print(f"Total queries: {total_queries} (of {total_queries_available} available)")
print(f"Workers: {MAX_WORKERS} | Model: {MODEL} | Filter: {FILTER_SERVICES} (top_k={TOP_K})")

Loaded 2 benchmark sets (total available: 2)
Total queries: 50 (of 157 available)
Workers: 10 | Model: deepseek-ai/DeepSeek-V4-Pro | Filter: True (top_k=5)


## Prompt Template

In [3]:
PROMPT_TEMPLATE = '''You are an expert software engineer performing REST service composition.

You are given:


A natural-language task description.
One or more REST API specifications (OpenAPI/Swagger).


Your job is to write a single self-contained Python script that fulfills the task by calling the necessary endpoints, in the correct order, passing data from earlier responses into later requests as required.

Output contract


Respond with raw Python source code ONLY. Your entire response must be directly executable by a Python interpreter with no edits.
The response must begin with an import statement (e.g. import requests). Do not emit markdown, code fences (```), backticks, language tags, comments, docstrings, prose, or trailing notes — nothing but code.
Import requests and define exactly one function named compose. Place all request logic inside compose. Do NOT call compose.
compose must return the final result that answers the task.


Composition rules


Use ONLY endpoints, HTTP methods, paths, parameters, and fields defined in the provided specifications. Do not invent endpoints, parameters, response fields, or hosts.
Build each request URL by joining the base URL from the spec\'s servers field with the operation path. If several servers are listed, use the first.
Place parameters exactly as the spec defines them: substitute path parameters into the URL, pass query parameters via params=, headers via headers=, and request bodies via json= (or data= for form bodies). Use the exact parameter and field names from the spec.
If the spec defines a security scheme (API key, bearer token, etc.), include it where the spec requires it (header or query). When no concrete value is supplied in the task, use a clearly named placeholder constant (e.g. API_KEY = "<API_KEY>").
Parse JSON responses with .json(), extract the specific fields you need, and feed them into subsequent calls. Chain calls so each step\'s output drives the next.
Iterate when the task requires processing a collection; otherwise issue each required call once.
Call only the endpoints strictly required to satisfy the task. Do not call supplementary endpoints whose output is not used as input to a later step or as the final result. When two endpoints seem relevant to the same task step, pick the one whose description most directly matches — do not call both.


Code-quality rules (the output is statically analyzed)


Use only the requests library and the Python standard library. No other third-party imports.
Every name must be defined before use. No undefined references, no unused imports or variables, no placeholders like ... or TODO.
Add type annotations to the compose signature and its return type, and to local variables where the type is clear. Import every typing symbol you reference (from typing import Any). Annotate decoded JSON as Any or dict[str, Any] rather than guessing concrete shapes.
Write valid, parseable Python with explicit, deterministic control flow.


Output shape (format example only — do NOT copy its logic or endpoints)

import requests
from typing import Any

def compose() -> Any:
    base_url = "https://api.example.com"
    first = requests.get(f"{{base_url}}/items", params={{"limit": 1}}).json()
    item_id = first["data"][0]["id"]
    detail = requests.get(f"{{base_url}}/items/{{item_id}}").json()
    return detail

Task

{query}

Source

{services_block}
'''


## Initial LLM Call

In [4]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

tasks = []
for benchmark in benchmark_sets:
    if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
        break
    for query_index, query in enumerate(benchmark['queries'], start=1):
        if QUERY_LIMIT is not None and len(tasks) >= QUERY_LIMIT:
            break
        tasks.append((benchmark, query_index, query))

def _call_initial(args):
    benchmark, query_index, query = args
    services = (
        filter_services(benchmark['services'], query['query'], top_k=TOP_K)
        if FILTER_SERVICES else benchmark['services']
    )
    prompt = build_prompt(services, query['query'], PROMPT_TEMPLATE)
    t0 = time.time()
    generated, usage = call_llm(prompt, MODEL, '')
    elapsed = time.time() - t0
    generated += '\n\ncompose()'
    return {
        'query_index': query_index,
        'sector_name': benchmark['name'],
        'query': query,
        'prompt': prompt,
        'generated': generated,
        'service_files': benchmark.get('service_files', []),
        'model': MODEL,
        '_elapsed': elapsed,
        '_usage': usage,
    }

sector_results = []
call_times = []
run_start = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_call_initial, t): t for t in tasks}
    for future in as_completed(futures):
        result = future.result()
        sector_results.append(result)
        call_times.append(result['_elapsed'])

        done = len(sector_results)
        avg_t = sum(call_times) / len(call_times)
        remaining = total_queries - done
        eta_s = remaining * avg_t / min(remaining, MAX_WORKERS) if remaining else 0
        usage = result['_usage']
        tok_str = (f"prompt={usage['prompt_tokens']} comp={usage['completion_tokens']}"
                   if usage else "tokens=n/a")
        print(
            f"[{done:>3}/{total_queries}] [{result['sector_name']}] Q{result['query_index']}"
            f" | {result['_elapsed']:.1f}s | {tok_str}"
            f" | avg {avg_t:.1f}s | ETA ~{eta_s:.0f}s"
        )

sector_results.sort(key=lambda r: (r['sector_name'], r['query_index']))
print(f"\nDone. Total wall time: {time.time() - run_start:.1f}s")

[  1/50] [spotify] Q4 | 12.7s | prompt=33271 comp=133 | avg 12.7s | ETA ~62s
[  2/50] [spotify] Q2 | 21.0s | prompt=33511 comp=275 | avg 16.9s | ETA ~81s
[  3/50] [spotify] Q10 | 23.1s | prompt=34111 comp=280 | avg 18.9s | ETA ~89s
[  4/50] [spotify] Q3 | 27.2s | prompt=34965 comp=414 | avg 21.0s | ETA ~97s
[  5/50] [spotify] Q8 | 30.5s | prompt=33916 comp=394 | avg 22.9s | ETA ~103s
[  6/50] [spotify] Q9 | 33.5s | prompt=33583 comp=530 | avg 24.7s | ETA ~109s
[  7/50] [spotify] Q5 | 39.0s | prompt=33755 comp=170 | avg 26.7s | ETA ~115s
[  8/50] [spotify] Q7 | 41.2s | prompt=33599 comp=223 | avg 28.5s | ETA ~120s
[  9/50] [spotify] Q12 | 28.1s | prompt=33672 comp=334 | avg 28.5s | ETA ~117s
[ 10/50] [spotify] Q1 | 51.6s | prompt=33909 comp=256 | avg 30.8s | ETA ~123s
[ 11/50] [spotify] Q13 | 29.1s | prompt=34900 comp=316 | avg 30.6s | ETA ~120s
[ 12/50] [spotify] Q6 | 61.5s | prompt=33594 comp=465 | avg 33.2s | ETA ~126s
[ 13/50] [spotify] Q15 | 62.6s | prompt=33509 comp=168 | avg 35.5

## Evaluate

In [5]:
for result in sector_results:
    initial_metrics = evaluate(result['generated'], result['query'].get('endpoints', []))
    result['initial_metrics'] = initial_metrics
    print_metrics(initial_metrics, f"Initial evaluation - Query {result['query_index']}")

Initial evaluation - Query 1
  Precision: 0.50
  Recall:    0.50
  F1:        0.50
  Extracted: ['GET /me', 'GET /search', 'POST /playlists//tracks', 'POST /users//playlists']
  Expected:  ['GET /me', 'GET /search', 'POST /playlists/{playlist_id}/tracks', 'POST /users/{user_id}/playlists']
  Missing:   ['POST /playlists/{playlist_id}/tracks', 'POST /users/{user_id}/playlists']
  Extra:     ['POST /playlists//tracks', 'POST /users//playlists']
Initial evaluation - Query 2
  Precision: 0.67
  Recall:    0.67
  F1:        0.67
  Extracted: ['GET /albums/', 'GET /search', 'POST /me/player/queue']
  Expected:  ['GET /albums/{id}/tracks', 'GET /search', 'POST /me/player/queue']
  Missing:   ['GET /albums/{id}/tracks']
  Extra:     ['GET /albums/']
Initial evaluation - Query 3
  Precision: 0.50
  Recall:    0.67
  F1:        0.57
  Extracted: ['GET /me', 'GET /me/playlists', 'GET /search', 'POST /playlists//tracks']
  Expected:  ['GET /me/playlists', 'GET /search', 'POST /playlists/{playlist_

## Save Outputs

In [6]:
from datetime import datetime
importlib.reload(output_module)
run_name = datetime.now().strftime('%Y-%m-%d_%H-%M-%S') + "_baseline"
outdir = output_module.make_output_dir(run_name)
for r in sector_results:
    output_module.write_query_output(outdir, r)

run_config = {
    'benchmark_dir': str(BENCHMARK_DIR),
    'benchmark_types': BENCHMARK_TYPES,
    'benchmark_limit': BENCHMARK_LIMIT,
    'query_limit': QUERY_LIMIT,
    'baseline': True,
    'model': MODEL,
    'filter_services': FILTER_SERVICES,
    'top_k': TOP_K if FILTER_SERVICES else None,
}
output_module.write_overall_summary(outdir, sector_results, run_config)
print('Wrote outputs to', outdir)

Wrote outputs to output/2026-06-30_13-44-39_baseline


## Summary

In [7]:
print_evaluation_summary([result['initial_metrics'] for result in sector_results], "Initial Evaluation Summary")

Initial Evaluation Summary
  Average Precision: 0.59
  Average Recall:    0.67
  Average F1:        0.61
  Avg. Missing Endpoints: 1.18
  Avg. Extra Endpoints:   1.76
  Correct Compositions: 11/50 (22.0%)
